# Synthea OMOP 1k — Initial Exploration

This notebook connects to the local DuckDB database built from the Synthea OMOP 1k cohort and runs a few basic analyses:

- Inspect OMOP tables (person, visit_occurrence, condition_occurrence, etc.)
- Compute simple cohort counts
- Demonstrate an example of "treatment gap" style analysis (patients with conditions but no drugs)

> Prerequisites:
> - `bash 01-initial-notebook/download_synthea_omop.sh`
> - `python 01-initial-notebook/load_synthea_duckdb.py`
> - `pip install duckdb`

In [1]:
import duckdb
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "01-initial-notebook" else Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "synthea1k.duckdb"

print(f"Using DuckDB database: {DB_PATH}")
con = duckdb.connect(DB_PATH.as_posix())

# List available OMOP-like tables
con.execute("SELECT table_name FROM duckdb_tables() ORDER BY table_name").df()

Using DuckDB database: /Users/isabelwu/Desktop/GSK DATA INTEGRATION/data/synthea1k.duckdb


,table_name
0,condition_era
1,condition_occurrence
2,drug_era
3,drug_exposure
4,measurement
5,observation
6,observation_period
7,person
8,procedure_occurrence
9,visit_occurrence


In [2]:
# Quick sanity checks on core OMOP tables

person_count = con.execute("SELECT COUNT(*) AS n_person FROM person").df()
visit_count = con.execute("SELECT COUNT(*) AS n_visit FROM visit_occurrence").df()
condition_count = con.execute("SELECT COUNT(*) AS n_condition FROM condition_occurrence").df()

display(person_count)
display(visit_count)
display(condition_count)

,n_person
0,1130


,n_visit
0,32153


,n_condition
0,7900


In [3]:
# Example "treatment gap" style analysis (generic)
# Patients with at least one condition record but no drug_exposure at all.

query = """
WITH condition_patients AS (
    SELECT DISTINCT person_id
    FROM condition_occurrence
),
drug_patients AS (
    SELECT DISTINCT person_id
    FROM drug_exposure
)
SELECT
    (SELECT COUNT(*) FROM person) AS total_persons,
    (SELECT COUNT(*) FROM condition_patients) AS persons_with_conditions,
    (SELECT COUNT(*) FROM drug_patients) AS persons_with_drugs,
    (SELECT COUNT(*) FROM condition_patients WHERE person_id NOT IN (SELECT person_id FROM drug_patients))
        AS condition_only_persons
"""

con.execute(query).df()

,total_persons,persons_with_conditions,persons_with_drugs,condition_only_persons
0,1130,1097,1121,0


## HBV-Style Care Cascade (Testing → Diagnosis → Treatment)

Simulates the WHO HBV cascade: **Testing** (lab) → **Diagnosis** (condition) → **Treatment** (drug).  
Synthea has no HBV-specific concepts; we use measurement/condition/drug as proxies.

In [4]:
# HBV-style cascade: Testing → Diagnosis → Treatment (sequential funnel)
cascade_query = """
WITH cohort AS (SELECT person_id FROM person),
     tested AS (SELECT DISTINCT person_id FROM measurement),
     diagnosed AS (
         SELECT DISTINCT t.person_id FROM tested t
         WHERE t.person_id IN (SELECT person_id FROM condition_occurrence)
     ),
     treated AS (
         SELECT DISTINCT d.person_id FROM diagnosed d
         WHERE d.person_id IN (SELECT person_id FROM drug_exposure)
     )
SELECT
    (SELECT COUNT(*) FROM cohort) AS total_persons,
    (SELECT COUNT(*) FROM tested) AS tested,
    (SELECT COUNT(*) FROM diagnosed) AS diagnosed,
    (SELECT COUNT(*) FROM treated) AS treated,
    (SELECT COUNT(*) FROM tested t WHERE t.person_id NOT IN (SELECT person_id FROM diagnosed))
        AS gap_tested_not_diagnosed,
    (SELECT COUNT(*) FROM diagnosed d WHERE d.person_id NOT IN (SELECT person_id FROM treated))
        AS gap_diagnosed_not_treated
"""
cascade = con.execute(cascade_query).df()
display(cascade)

,total_persons,tested,diagnosed,treated,gap_tested_not_diagnosed,gap_diagnosed_not_treated
0,1130,1120,1096,1096,24,0


In [5]:
# Cascade funnel visualization (simple bar)
import pandas as pd
row = cascade.iloc[0]
funnel = pd.DataFrame({
    "Stage": ["Total", "Tested", "Diagnosed", "Treated"],
    "Count": [row["total_persons"], row["tested"], row["diagnosed"], row["treated"]]
})
display(funnel)
print(f"Gap (tested → diagnosed): {row['gap_tested_not_diagnosed']} | Gap (diagnosed → treated): {row['gap_diagnosed_not_treated']}")

,Stage,Count
0,Total,1130
1,Tested,1120
2,Diagnosed,1096
3,Treated,1096


Gap (tested → diagnosed): 24 | Gap (diagnosed → treated): 0


In [6]:
# Close connection when done
con.close()